# MobileViTv2-0.75x 微调训练（ImageNet-1k 预训练 → 10 类，激活函数换为 ReLU）

**按要求做的调整**

1. 宽度 `WIDTH_MULTIPLIER = 0.75`，与纯权重文件 `mobilevitv2-0.75.pt`（ImageNet-1k，269 个张量，2,877,698 个参数）一致；
2. 激活函数由 `swish`(SiLU) 改为 `relu`：`build_opts` 里同时设置
   `model.activation.name` 与 `model.classification.activation.name`。ReLU / Swish 都没有可学习参数，
   所以 state_dict 的键完全不变（已实测：换激活后 269 个键一一对应），ImageNet 权重可以原样加载；
3. 类别数由数据集自动读取（`train_p` 下 10 个类别目录 c0~c9，共 22,424 张 256×256 RGB 图）；
4. 用 `PRETRAINED_WEIGHTS` 初始化：加载除 1000 类分类头以外的全部 backbone 权重（267/269），
   10 类分类头按 cvnets 的 `trunc_normal` 重新初始化。

**修复的原脚本问题**

| 问题 | 说明 |
| --- | --- |
| 预训练权重根本加载不上 | 改成 10 类后 `classifier.1.weight` 由 (1000,384) 变 (10,384)，实测 `load_state_dict` 无论 `strict` 与否都会抛 `RuntimeError`；现在先按形状筛出可加载的键，再 `strict=True` 加载，并校验"除分类头外没有别的键被跳过" |
| 余弦退火实际失效 | 原 `T_MAX_ITERS=1000` < 预热步数 3366，`CosineAnnealingLR(T_max=1000-3366)` 收到负数；现在 `T_MAX_ITERS=None` 时自动取 `EPOCHS × 每轮步数`，不合法直接报错 |
| EMA 起点错误 | 原脚本在加载权重**之前**创建 EMA，且"纯权重"分支没有重建 EMA，导致评估/保存用的 EMA 从随机初始化开始；现在权重加载或恢复之后再创建 EMA |
| MixUp/CutMix 分支缺少标签平滑 | 混增强分支用裸 `F.cross_entropy`，与主 `criterion` 的 `label_smoothing` 不一致 |
| 训练准确率恒为 -1 | 增强开启时直接返回 -1；现在用"未增强的 batch"统计并明确标注为近似值 |
| 检查点信息不足 | 检查点现在带 `config`（类别名、宽度、激活、T_max、数据集路径等），推理端可据此交叉校验类别顺序 |
| 其它 | 重复 import cvnets、`t_max_value` 重复计算、`warmup_steps` 为 0 时除零、断点续训与保存路径撞车、可能覆盖 ImageNet 预训练权重 |

**运行顺序**：依次运行下面 3 个代码单元即可；`main()` 在最后一个单元里，改超参后重跑该单元。

数据集：`C:\transformerFPGA\MobileViT\Dataset\Driver\train_p`。原图已经是 256×256（能被 32 整除），
因此**不做 Resize**，训练/验证/推理三处预处理保持一致（Resize(可选) → ToTensor → ImageNet 归一化）。

---

## 本次修订（按代码审阅意见）

审阅提出的问题逐条核对后确认成立，已全部修正：

| # | 问题 | 修正 |
| --- | --- | --- |
| 1 | **EMA 检查点不能正确续训**：`model_state_dict` 里存的是 EMA 权重，而 `optimizer_state_dict` 来自原始模型，恢复时把 EMA 参数装进训练模型再套原优化器状态，两者不对应 | 统一走 `build_checkpoint()`：`model_state_dict` 存**原始权重**（续训用，与优化器状态配套），`ema_model_state_dict` 存 **EMA 权重**（推理用），并写入 `inference_weights` 指明推荐用哪一份；续训按 `model_state_dict` 恢复，EMA 在恢复后重新播种。想从 EMA 续训可设 `RESUME_FROM_EMA=True`（会明确告警） |
| 2 | **追加训练可能让学习率回升**：`CosineAnnealingLR` 在步数超过 `T_max` 后 cos 项重新变大，续训沿用已走完的调度器就会看到学习率上涨 | 换成单个 `LambdaLR`（`make_lr_lambda`），退火进度用 `min(1.0, …)` 夹住，超过周期后恒为 `ETA_MIN`；恢复时若发现已越过周期会明确提示（要重新退火就设 `RESUME_RESET_SCHEDULER=True`） |
| 3 | **重置调度器后首个优化步跳过预热起点**：先建调度器再把 `lr` 手动改成 `target_lr`，第一步直接用满学习率 | 先写好基准学习率与追加步数再建调度器，保留调度器初始化算出的当前学习率（= 预热起点）；重置周期改用 `ADDITIONAL_EPOCHS × 每轮步数`（原来错用 `EPOCHS`）；预热步数放不下时自动收缩到周期的 10% 并提示 |
| 4 | **只开一种增强时概率参数失效**：`elif mixup_enable: apply_mixup = True` 会让 `MIXUP_PROB=0.01` 形同虚设 | 单开也按各自概率抽样；`main()` 校验概率范围，并提示 `MIXUP_PROB + CUTMIX_PROB > 1` 的情况 |
| 5 | **梯度累积最后一组不足时梯度被多除** | 按当前累积组实际 batch 数归一化：`group_size = min(grad_accum_steps, num_batches - group_start)` |
| 6 | **"最佳准确率"与保存的最佳模型可能不是同一轮**：按最低验证损失保存，`best_val_acc` 却是历史最高准确率 | 检查点同时记录本轮 `val_loss`/`val_acc` 与判据/历史值 `best_loss`/`best_epoch`/`best_val_acc`，保存与收尾汇总都把两者分开打印 |
| 7 | **续训前仍然加载预训练文件**：白做一次加载，且预训练文件缺失会让续训直接失败 | `RESUME=True` 时跳过预训练初始化并说明原因 |

顺带清理的冗余：检查点字段只在 `build_checkpoint()` 里构造（早停保存与每轮保存共用）；续训状态统一走 `EarlyStopping.load_state()`；EMA 去掉未使用的 `*args/**kwargs`，`ema_momentum` 类型改为 `float`；混合增强损失直接用 `criterion`（不再单独传 `label_smoothing`）；类别计数改用 `collections.Counter`；删掉不再使用的导入（`Tuple`、`torch.nn.functional`）。

> `t.txt` 是本文件后两个代码单元的拼接，缺少第 1 个单元的导入以及 `build_opts` / `build_transform` / `load_weights` 等定义，所以它单独无法运行；在笔记本里按单元顺序执行即可。

> 检查点体积：`CHECKPOINT_INCLUDE_EMA=True` 时每个检查点同时含原始权重与 EMA 权重（约 40MB／轮，30 轮约 1.2GB）。只想留最佳模型就把 `CHECKPOINT_EVERY_EPOCH` 设为 `False`；关掉 `CHECKPOINT_INCLUDE_EMA` 只省掉 EMA 那一份权重（优化器状态仍在，实际节省不到一半）。

---

## 第二轮修订（按第二次审阅意见）

| # | 问题 | 修正 |
| --- | --- | --- |
| 1 | **续训没有恢复 EMA**：检查点里存了 `ema_model_state_dict`，恢复时却只用原始权重重建 EMA，历史滑动平均被丢弃，验证指标会突变 | 续训时把 EMA 权重**严格加载回 EMA 模型**（形状/键不匹配会告警并退回重建），验证指标连续 |
| 2 | **缺失的主干权重发现不了**：`model_state.update(loadable)` 先把所有键补齐，`strict=True` 就失去了检查"文件缺键"的能力（只给分类头也能通过） | `load_weights()` 改为双向检查：文件**缺少**模型需要的非分类头参数 → 报错；文件里有模型用不上的非分类头键 → 报错。局部测试：只给分类头、缺一个主干键都会报错，完整 10 类权重 269 个全部加载 |
| 3 | **恢复调度器状态 ≠ 恢复原曲线**：`make_lr_lambda` 的闭包参数不会被 `LambdaLR.state_dict()` 保存，配置一变曲线就变 | 检查点新增 `scheduler_config`（周期、预热步数、基准/起点/终点学习率、每轮步数）；续训时**先按保存的配置重建曲线，再加载调度器状态**。局部测试：恢复后 20 步与原曲线逐步一致，按"当前错误配置"恢复则偏到别的值 |
| 4 | **覆盖基准学习率会把起点/终点等比缩放**：原来返回"相对 `MAX_LR` 的倍率"，基准换 1e-4 后起点变 5e-7、终点 5e-11 | 改为**绝对学习率语义**：预热从 `WARMUP_START_LR` 升到基准，再余弦降到 `ETA_MIN`，与基准无关。局部测试：基准 1e-4 时起点 1e-6、峰值 1e-4、终点与周期后都是 1e-10 |
| 5 | **"最佳轮准确率"可能配错轮次**：`best_epoch` 来自最低损失轮，`val_acc` 却来自当前轮 | 新增 `best_loss_val_acc`（最低损失**那一轮**的准确率）并随检查点保存/恢复（旧格式回退到 `val_acc`）。局部测试：第 3 轮最佳、第 10 轮更差时三者互不串台 |
| 6 | **续训后不保证保存路径里真有最佳模型**：损失一直没改善就不写文件，收尾却报告"最佳模型"并去加载（可能不存在或读到旧文件） | 新增 `EarlyStopping.saved_this_run`：区分"历史最佳"与"本次是否写入"；没写入就明确报告并跳过收尾自检；保存路径已存在时先提示；另加 `RESUME_RESET_BEST_LOSS=True` 让追加训练重新判定最佳并写入新文件 |
| 7 | **关闭 `CHECKPOINT_INCLUDE_EMA` 时验证指标与保存权重不对应** | 最佳模型**始终**保存"验证时用的那份权重"（EMA 开启时含 EMA）；该开关只影响每轮检查点 |
| 8 | **梯度累积未按样本数加权**：尾组小 batch 与整批等权 | 组内按"该 batch 样本数 / 组内总样本数"加权（`batch_sample_counts()`），组内权重和为 1；`GRAD_ACCUM_STEPS=1` 时与原来完全一致。局部测试：30 样本、批 8、累积 2 → 尾组权重 8/14 与 6/14 |
| 9 | **`RESIZE=256` 不能保证尺寸一致**：整数只固定短边 | 文档与提示统一改成"要固定尺寸请写 `(256, 256)`"，`build_transform()` 对整数写法给出提示 |

冗余清理：删掉只赋值不读取的 `last_val_acc`；`include_ema` 不再层层传递（最佳模型保存恒为 True）；`rand_bbox` 的 `W/H` 命名改为 `row/col`（行为不变）；`run_meta` 记录**实际使用**的调度配置，并保留配置原始值便于对照。

> 以上 9 项都做了局部逻辑测试（不读数据、不训练、不用 GPU，共 9 项全部通过）；语法与静态检查（JSON、逐单元 compile、pyflakes）同样通过。

---

## 第三轮修订（按第三次审阅意见）

| # | 问题 | 修正 |
| --- | --- | --- |
| 1 | **调度配置恢复不完整**：保存了 `start_lr`/`min_lr`，恢复时却只读周期与基准，改了这两个配置续训曲线就变 | 恢复时把 `start_lr` / `min_lr` 一起读回并显式传给 `build_scheduler()`；只有主动 `RESUME_RESET_SCHEDULER=True` 时才使用当前配置 |
| 2 | **纯权重续训忽略 `RESUME_LR` / `RESUME_RESET_SCHEDULER`**，且退火周期仍按 `EPOCHS` 算，与实际只练 `ADDITIONAL_EPOCHS` 轮不匹配 | 把"本次学习率与退火周期"的定稿逻辑从完整检查点分支里提出来，两种续训模式统一处理；纯权重或主动重置时，周期 = `ADDITIONAL_EPOCHS × 每轮步数`（显式设了 `T_MAX_ITERS` 则优先用它） |
| 3 | **"历史最佳仍在续训文件里"不一定成立**（第 30 轮的检查点只有第 30 轮权重） | 检查点新增 `best_checkpoint_path`，记录历史最佳权重当时写到了哪个文件；续训与收尾报告改为如实引用它（并检查文件是否仍在），不再声称最佳权重在 `RESUME_PATH` 里 |
| 4 | **旧格式 `best_loss_val_acc` 兜底不准**：直接用 `val_acc` 会把当前轮准确率算到最佳轮 | 只有 `epoch == best_epoch` 时才允许用 `val_acc` 兜底，否则记为未知（报告显示"未知"） |
| 5 | **"已有文件不会被覆盖"的提示与实际行为不符** | 按实情分两种提示：续训且未重置最佳 → "只有更好才会覆盖"；从零开始 → "基准 val_loss=inf，第一轮就会覆盖" |
| 6 | **恢复失败后的回退没有真正重建对象** | 调度器恢复失败 → 重新 `build_scheduler()`；EMA 恢复失败 → 重新 `EMA(model, ...)`，回退状态确定 |
| 7 | 调度器被创建两次（创建后又因续训再建一次） | 先把配置定稿，再**只创建一次**调度器 |
| 8 | 组内样本总数被重复计算 | 同一个累积组只求一次，进组时缓存 |
| 9 | 注释不准确（采样前提 / "等价于大 batch"） | 改为"默认批处理方式 + `drop_last=False`，`shuffle` 无关"；并注明梯度累积**不等于**真正的大 batch（模型含 BN/Dropout，两者不会逐位一致） |
| 10 | `GRAD_ACCUM_STEPS` 未校验；`WARMUP_EPOCHS=0` 被强制成"至少预热 1 步" | 新增 `GRAD_ACCUM_STEPS` 必须为 ≥1 整数的校验；预热步数改为 `int(round(WARMUP_EPOCHS × 每轮步数))`，`0` 真的是 0 步；`Counter` 导入移到单元 1 |

### 实跑记录（8 张 256×256 图 / 2 类 / 各 1 轮）

* **运行 A（全新训练）**：`预热步数: 0（WARMUP_EPOCHS=0，不预热）` → 第 1 步 lr = `2.000e-04`（就是基准值，确实没预热）；训练 3 步、验证 2 张、保存 `best.pth` 与 `ckpt_1.pth`；收尾自检对 `model_state_dict` 与 `ema_model_state_dict` 都严格加载通过。
* **运行 B（续训）**：跳过预训练初始化 → `采用检查点保存的调度配置: 周期 3 步, 预热 0 步, 基准 0.0002, 起点 1e-06, 终点 1e-10` → `历史最佳权重当时保存在: ...best.pth（文件仍在）` → `已恢复调度器状态: 第 3 步, 当前学习率 1.000e-10`，并提示"已越过退火周期，学习率不会回升" → `EMA 已从检查点恢复` → 正常训练第 2 轮、保存新检查点、自检通过。

---

## 第四轮修订（按第四次审阅意见）

| # | 问题 | 修正 |
| --- | --- | --- |
| 1 | **`best_checkpoint_path` 记的是本次输出路径，而不是历史最佳的实际位置**：续训一直没改善时，每轮检查点会谎称历史最佳在本次的 `SAVE_PATH`（该文件可能根本不存在） | 这个字段改由 `EarlyStopping` 自己维护：`load_state()` 时**继承**检查点里的值，只有**真正写出**新的最佳模型时才更新为本次路径；`build_checkpoint()` 与所有报告都改读 `early_stopping.best_checkpoint_path` |
| 2 | **"周期 > 预热"的校验发生在最终配置定稿之前**，会用全新训练的 `EPOCHS/WARMUP_EPOCHS` 误拦合法的续训（例如 `EPOCHS=2`、`WARMUP_EPOCHS=3`，但检查点里的周期原本合法） | 基础检查（周期为正、预热非负）留在原处；"周期 > 预热"移到**最终调度配置定稿之后、创建调度器之前**判断 |
| 3 | **极短追加训练可能出现"预热 = 整个周期"**（周期 1 步、预热 1 步时，学习率走完这一步才刚到峰值，永远退不到终点） | 统一保证 `0 <= 预热 < 周期`：续训时自动收缩预热并提示；全新训练的配置不合理时直接报错；`make_lr_lambda()` 内部也强制该不变式 |

顺带清理：`config['scheduler']` 与顶层 `scheduler_config` 重复 → 只保留顶层作为正式字段（`config` 里两项注明"按超参推算的原值，仅用于对照"）；设置学习率的循环抽成 `reset_scheduler()`，创建与异常回退共用；注明 `LR` 是唯一入口（`MAX_LR` 只是峰值别名）；精简若干描述旧版错误的注释。**`build_opts` 的选项一个都没删**（必须与所装 cvnets 版本对应）。

### 实跑记录（8 张 256×256 图 / 2 类）

* **运行 A（全新训练 1 轮）**：`预热步数: 2`、周期 3 步（`WARMUP_EPOCHS=0.5`）；保存 `best.pth` 与 `ckpt_1.pth`；检查点里 `scheduler_config` 记录了起点/终点/预热/周期，`best_checkpoint_path` = 本次 `best.pth`。
* **运行 B（换新输出路径续训 1 轮）**：`SAVE_PATH=new_best.pth`、从 `ckpt_1.pth` 续训 → 结束后核对不变式：`ckpt_resume_2.pth['best_checkpoint_path']` 等于**真正写出最佳权重的那个文件**（本次有新最佳 → `new_best.pth`；没改善 → 仍是 A 的 `best.pth`），不再指向未生成的文件。
* **运行 C（极短追加：周期 = 1 步）**：`BATCH_SIZE=6`（每轮只有 1 个优化步）+ `RESUME_RESET_SCHEDULER=True` + `ADDITIONAL_EPOCHS=1` → 打印 `预热步数 1 不小于退火周期 1，已收缩为 0`，检查点里 `warmup_steps=0, horizon=1`，第 1 步学习率就是基准值 `2.000e-04`（不再停在峰值不动）。
* **运行 D（全新训练配置不合理）**：`EPOCHS=1` + `WARMUP_EPOCHS=1`（周期=预热=每轮步数）→ 直接报错并提示"把 WARMUP_EPOCHS 调小或 EPOCHS/T_MAX_ITERS 调大"，不会再生成一条"整段停在峰值"的曲线。


In [1]:
# ==================== 依赖与模型构建选项 ====================
import argparse
import math
import os
import random
from collections import Counter
from copy import deepcopy
from typing import Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm import tqdm

try:
    from cvnets.models.classification.mobilevit_v2 import MobileViTv2
except ImportError as exc:  # 当前内核没有 cvnets 时给出明确提示
    raise ImportError(
        "当前内核找不到 cvnets，请把本笔记本的内核切到 conda 的 py312 环境，"
        "并在该环境中安装 ml-cvnets（例如 pip install -e C:/transformerFPGA/MobileViT/ml-cvnets）。"
    ) from exc


def build_opts(num_classes: int, width_multiplier: float = 0.75, act_name: str = "relu") -> argparse.Namespace:
    """构造 MobileViTv2 需要的 opts。

    两个容易踩的坑（已实测确认）：
    1) 这里直接调用 ``MobileViTv2(opts)``，而不是 ``MobileViTv2.build_model(opts)``。
       ``model.classification.activation.*`` 只在 ``build_model()`` 内部才会被复制到
       ``model.activation.*``，所以**真正生效的是 ``model.activation.name``**。
       为了两种构建方式行为一致，两处都写。
    2) 激活函数从 swish 换成 relu 不会改变任何 state_dict 键（两者都没有参数），
       因此 ImageNet 预训练权重可以原样加载，不需要做键名映射。
    """
    opts = argparse.Namespace()
    attrs = {
        "model.classification.n_classes": num_classes,
        "model.classification.gradient_checkpointing": False,
        "model.classification.enable_layer_wise_lr_decay": False,
        "model.classification.layer_wise_lr_decay_rate": 1.0,
        "model.classification.freeze_batch_norm": False,
        "model.classification.pretrained": None,
        "model.classification.finetune_pretrained_model": False,
        "model.classification.n_pretrained_classes": None,
        "model.classification.classifier_dropout": 0.1,
        "model.classification.activation.name": act_name,
        "model.classification.activation.inplace": False,
        "model.classification.activation.neg_slope": 0.1,
        "model.activation.name": act_name,
        "model.activation.inplace": False,
        "model.activation.neg_slope": 0.1,
        "model.normalization.name": "batch_norm",
        "model.normalization.momentum": 0.1,
        "model.normalization.eps": 1e-5,
        "model.normalization.groups": 32,
        "model.layer.conv_init": "kaiming_normal",
        "model.layer.linear_init": "trunc_normal",
        "model.layer.pretrained": None,
        "model.layer.global_pool": "mean",
        "model.classification.mitv2.width_multiplier": width_multiplier,
        "model.classification.mitv2.attn_dropout": 0.1,
        "model.classification.mitv2.ffn_dropout": 0.1,
        "model.classification.mitv2.dropout": 0.2,
        "model.classification.mitv2.attn_norm_layer": "layer_norm_2d",
        "model.layer.deit": None,
        "common.enable_coreml_compatible_module": False,
        "model.segmentation.n_classes": None,
        "common.regnet.quantize": False,
        "common.mixed_precision": False,
    }
    for k, v in attrs.items():
        setattr(opts, k, v)
    return opts


def build_transform(resize=None):
    """训练 / 验证 / 推理必须使用完全一致的预处理。

    本数据集全部是 256×256（能被下采样倍数 32 整除），所以 ``resize=None`` 即可。

    ``resize`` 的语义与 torchvision 一致：**整数只把"短边"缩放到该值**，长宽比不同的图片
    结果尺寸仍然不同，凑不成一个 batch；要真正固定尺寸，请传元组，例如 ``(256, 256)``。
    """
    ops = []
    if resize is not None:
        if isinstance(resize, int):
            print(f"提示: Resize({resize}) 只固定短边，长宽比不同的图片尺寸仍会不一致；"
                  f"要固定尺寸请把 RESIZE 写成 ({resize}, {resize})")
        ops.append(transforms.Resize(resize))
    ops += [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
    return transforms.Compose(ops)


def extract_state_dict(ckpt) -> Dict[str, torch.Tensor]:
    """从"纯权重文件"或"完整检查点"中取出 state_dict，并去掉 ``module.`` 前缀。"""
    if not isinstance(ckpt, dict):
        raise TypeError(f"无法识别的权重文件类型: {type(ckpt)}")

    if all(torch.is_tensor(v) for v in ckpt.values()):
        state = ckpt  # 纯权重（OrderedDict[str, Tensor]）
    else:
        state = None
        for key in ("model_state_dict", "ema_model_state_dict", "state_dict", "model", "module"):
            inner = ckpt.get(key)
            if isinstance(inner, dict) and all(torch.is_tensor(v) for v in inner.values()):
                state = inner
                print(f"  从键 '{key}' 中提取模型权重")
                break
        if state is None:
            raise ValueError(f"无法在权重文件中找到 state_dict，可用键: {list(ckpt.keys())}")

    return {k[7:] if k.startswith("module.") else k: v for k, v in state.items()}


def load_weights(model, path, expect_only_head_to_be_skipped=True) -> Dict:
    """把权重加载进模型，两个方向都检查：

    1. **文件里缺少模型需要的参数** —— 这些参数会保持随机初始化（等于从随机权重开始训练），
       除分类头外一律报错；
    2. **文件里有模型用不上的键** —— 键名/结构不匹配，除分类头外一律报错。

    为什么要按形状过滤：换成 10 类后 ``classifier.1.weight`` 从 (1000, 384) 变成 (10, 384)，
    实测 ``load_state_dict(strict=False)`` 在形状不匹配时同样报 RuntimeError，
    所以先挑出兼容的键，再用 ``strict=True`` 加载其余部分。
    """
    if not os.path.isfile(path):
        raise FileNotFoundError(f"找不到权重文件: {path}")

    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    run_meta = ckpt.get("config", {}) if isinstance(ckpt, dict) else {}
    state = extract_state_dict(ckpt)
    model_state = model.state_dict()

    loadable, skipped = {}, {}
    for k, v in state.items():
        if k not in model_state:
            skipped[k] = "模型中不存在该键"
        elif tuple(model_state[k].shape) != tuple(v.shape):
            skipped[k] = f"形状不匹配 {tuple(v.shape)} -> {tuple(model_state[k].shape)}"
        else:
            loadable[k] = v

    # 方向一：模型需要、但文件没给的参数（注意此时还不能 model_state.update(loadable)，
    # 否则所有键都被"补齐"，strict=True 就永远发现不了缺失）
    unloaded = [k for k in model_state if k not in loadable]
    unloaded_non_head = [k for k in unloaded if not k.startswith("classifier.")]
    unloaded_head = [k for k in unloaded if k.startswith("classifier.")]
    if unloaded_non_head:
        raise RuntimeError(
            f"权重文件缺少 {len(unloaded_non_head)} 个模型参数，它们会保持随机初始化，"
            f"训练结果不可信：{unloaded_non_head[:10]}"
            + (" ..." if len(unloaded_non_head) > 10 else "")
        )

    # 方向二：文件里有、模型用不上的键
    if expect_only_head_to_be_skipped:
        unexpected = [k for k in skipped if not k.startswith("classifier.")]
        if unexpected:
            raise RuntimeError(
                "除分类头之外还有权重没加载上，说明权重与模型结构不匹配，"
                f"请检查 width_multiplier / attn_norm_layer 等配置是否与预训练权重一致：{unexpected[:10]}"
            )

    model_state.update(loadable)
    model.load_state_dict(model_state, strict=True)

    print(f"权重加载: {path}")
    print(f"  已加载 {len(loadable)} 个张量（模型共 {len(model_state)} 个参数）"
          f"，跳过 {len(skipped)} 个文件里的键")
    if unloaded_head:
        print(f"  {len(unloaded_head)} 个分类头参数未加载，保持初始化值（微调时的预期行为）")
    for k, why in skipped.items():
        print(f"    跳过 {k}: {why}")

    if run_meta:
        print(f"  该权重自带配置: 类别数={run_meta.get('num_classes')} "
              f"宽度={run_meta.get('width_multiplier')} 激活={run_meta.get('activation')} "
              f"类别名={run_meta.get('class_names')}")
    return {"loaded": len(loadable), "skipped": skipped, "unloaded": unloaded, "config": run_meta}


In [2]:
# ==================== 工具类与训练/验证函数 ====================
class EMA(object):
    """对模型参数做指数滑动平均。

    Args:
        model: 需要做 EMA 的模型
        ema_momentum: 当前迭代权重的贡献比例，默认 0.0005
        device: 模型所在设备

    注意：EMA 必须在**加载完权重之后**再创建，否则 EMA 模型会停留在随机初始化状态；
    续训时如果检查点里存了 EMA 权重，应当把它加载回来（见 main 里的 resume_ema_state），
    而不是重新从原始权重开始累积。
    """

    def __init__(self, model: nn.Module, ema_momentum: float = 0.0005,
                 device: Optional[str] = "cpu") -> None:
        # 深拷贝一份模型用于累积滑动平均，并切到 eval 模式
        self.ema_model = deepcopy(model)
        self.ema_model.eval()
        self.momentum = ema_momentum
        self.device = device
        if device:
            self.ema_model.to(device=device)
        self.ema_has_module = hasattr(self.ema_model, "module")
        for param in self.ema_model.parameters():
            param.requires_grad = False

    def update_parameters(self, model):
        # 处理 DataParallel/DDP 包装造成的键名前缀差异
        has_module = hasattr(model, "module") and not self.ema_has_module
        with torch.no_grad():
            msd = model.state_dict()
            for k, ema_v in self.ema_model.state_dict().items():
                if has_module:
                    k = "module." + k
                model_v = msd[k].detach()
                if self.device:
                    model_v = model_v.to(device=self.device)
                if torch.is_floating_point(ema_v):
                    ema_v.copy_((ema_v * (1.0 - self.momentum)) + (self.momentum * model_v))
                else:
                    # 整数 buffer（如 num_batches_tracked）求平均没有意义，直接同步
                    ema_v.copy_(model_v)


def build_checkpoint(model, ema, optimizer, scheduler, epoch, val_loss, val_acc,
                     best_val_acc, early_stopping, run_meta,
                     scheduler_config=None, include_ema=True):
    """统一构造检查点（早停保存与每轮保存共用，避免两处字段写法不一致）。

    字段约定（续训与推理各取所需）：
    * model_state_dict     —— 原始训练模型的权重；**续训用它**，与 optimizer_state_dict 对应
    * ema_model_state_dict —— EMA 权重；**推理/部署优先用它**
    * inference_weights    —— 告诉推理脚本优先取哪一份权重
    * scheduler_config     —— 学习率曲线的完整参数（普通函数的闭包不会被 state_dict 保存，
                              不存这份配置就没法原样恢复曲线）
    * val_loss / val_acc   —— 这个检查点自己那一轮的验证指标
    * best_loss / best_epoch / best_loss_val_acc —— 早停判据（最低验证损失）及**那一轮**的准确率
    * best_val_acc         —— 历史最高验证准确率（可能来自别的轮次，别和 val_acc 混用）
    * best_checkpoint_path —— 历史最佳权重**实际**写在哪个文件（不是本次输出路径；续训时继承）
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'inference_weights': 'ema' if (ema is not None and include_ema) else 'model',
        'best_checkpoint_path': getattr(early_stopping, 'best_checkpoint_path', None),
        'optimizer_state_dict': optimizer.state_dict() if optimizer else None,
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'scheduler_config': scheduler_config or {},
        'val_loss': val_loss,
        'val_acc': val_acc,
        'best_val_acc': best_val_acc,
        'best_loss': early_stopping.best_loss,
        'best_epoch': early_stopping.best_epoch,
        'best_loss_val_acc': early_stopping.best_loss_val_acc,
        'early_stop_counter': early_stopping.counter,
        'config': run_meta or {},
    }
    if ema is not None and include_ema:
        checkpoint['ema_model_state_dict'] = ema.ema_model.state_dict()
    return checkpoint


class EarlyStopping:
    """按验证损失早停；验证损失下降时保存检查点。

    保存的检查点里**始终**包含用于验证的那份权重（开启 EMA 时就是 EMA 权重）与原始权重，
    这样"选择检查点的指标"和"检查点里的权重"始终对得上。
    """

    def __init__(self, patience=10, verbose=True, delta=0.0, path='best_model.pth'):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_loss = np.inf
        self.best_epoch = None
        self.best_loss_val_acc = None   # 最低验证损失那一轮的准确率
        self.best_checkpoint_path = None  # 历史最佳权重**实际**写到了哪个文件（续训时继承）
        self.early_stop = False
        self.saved_this_run = False     # 本次训练是否真的写过检查点

    def __call__(self, val_loss, val_acc, model, optimizer=None, scheduler=None, epoch=0,
                 best_val_acc=0.0, ema=None, run_meta=None, scheduler_config=None):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.best_epoch = epoch
            self.best_loss_val_acc = val_acc
            self.counter = 0
            self.save_checkpoint(model, optimizer, scheduler, epoch, val_loss, val_acc,
                                 best_val_acc, ema, run_meta, scheduler_config)
        else:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, model, optimizer=None, scheduler=None, epoch=0, val_loss=None,
                        val_acc=None, best_val_acc=0.0, ema=None, run_meta=None,
                        scheduler_config=None):
        # 这里始终 include_ema=True：验证用的是 EMA 权重，保存的也必须是它
        # 先记下"最佳权重这次写到了哪里"，再构造检查点，
        # 这样检查点里的 best_checkpoint_path 就是它自己的路径（顺序反了会记成上一层）
        self.best_checkpoint_path = self.path
        checkpoint = build_checkpoint(model, ema, optimizer, scheduler, epoch, val_loss, val_acc,
                                      best_val_acc, self, run_meta,
                                      scheduler_config=scheduler_config, include_ema=True)
        torch.save(checkpoint, self.path)
        self.saved_this_run = True
        if self.verbose:
            metrics = (f"val_loss={val_loss:.4f}, val_acc={val_acc:.2f}%"
                       if val_loss is not None and val_acc is not None else "验证指标未知")
            print(f"验证损失下降，保存检查点至 {self.path}"
                  f"（第 {epoch} 轮: {metrics}；历史最高 val_acc={best_val_acc:.2f}%；"
                  f"推理权重={checkpoint['inference_weights']}）")

    def load_state(self, checkpoint):
        """从检查点恢复早停状态（主流程统一调用本方法，避免两处各写一遍）。"""
        self.best_loss = float(checkpoint.get('best_loss', np.inf))
        self.counter = int(checkpoint.get('early_stop_counter', 0))
        self.best_epoch = checkpoint.get('best_epoch')
        # 旧格式检查点没有 best_loss_val_acc：只有"当前轮正好是最佳轮"时才能拿 val_acc 顶上，
        # 否则宁可为 None（报告里写"未知"），也不能把别的轮次的准确率张冠李戴。
        acc = checkpoint.get('best_loss_val_acc')
        if acc is None and checkpoint.get('epoch') == checkpoint.get('best_epoch'):
            acc = checkpoint.get('val_acc')
        self.best_loss_val_acc = acc
        # 历史最佳权重的实际位置要继承过来：本次输出路径可能压根没写过文件
        self.best_checkpoint_path = checkpoint.get('best_checkpoint_path')


def mixup_data(x, y, alpha=1.0):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def rand_bbox(size, lam):
    """返回矩形框在 (H, W) 两个空间维上的起止索引：row1, col1, row2, col2。"""
    h = size[-2]
    w = size[-1]
    cut_rat = np.sqrt(1. - lam)
    cut_h = int(h * cut_rat)
    cut_w = int(w * cut_rat)
    cy = np.random.randint(h)
    cx = np.random.randint(w)
    row1 = np.clip(cy - cut_h // 2, 0, h)
    col1 = np.clip(cx - cut_w // 2, 0, w)
    row2 = np.clip(cy + cut_h // 2, 0, h)
    col2 = np.clip(cx + cut_w // 2, 0, w)
    return row1, col1, row2, col2


def cutmix_data(x, y, alpha=1.0):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    row1, col1, row2, col2 = rand_bbox(x.size(), lam)
    mixed_x = x.clone()
    mixed_x[:, :, row1:row2, col1:col2] = x[index, :, row1:row2, col1:col2]
    lam = 1 - ((row2 - row1) * (col2 - col1) / (x.size(-1) * x.size(-2)))
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def batch_sample_counts(loader):
    """推断每个 batch 的样本数，用于梯度累积的按样本加权。

    适用范围：使用默认批处理方式的 DataLoader（``batch_size=…``、``drop_last=False``），
    ``shuffle`` 与否不影响结论；前提是能拿到 ``loader.batch_size`` 且每个 batch 的大小由它决定
    （默认 sampler + 默认 collate）。取不到 ``batch_size`` 时退化为所有 batch 等权。
    """
    num_batches = len(loader)
    batch_size = getattr(loader, 'batch_size', None)
    if not batch_size or num_batches == 0:
        return [1] * num_batches
    total = len(loader.dataset)
    counts = [batch_size] * num_batches
    remainder = total % batch_size
    if remainder:
        counts[-1] = remainder
    return counts


def train_one_epoch(model, loader, criterion, optimizer, scheduler, device, epoch, use_bf16,
                    mixup_enable=False, mixup_alpha=0.2, mixup_prob=0.5,
                    cutmix_enable=False, cutmix_alpha=1.0, cutmix_prob=0.5,
                    ema=None, grad_accum_steps=1, grad_clip_max_norm=5.0):
    """训练一轮。

    返回 (epoch_loss, epoch_acc, augmented_batches)：
    * epoch_acc 只统计**未做混合增强**的样本，增强开启时它是近似值（可能为 None）；
    * augmented_batches 用于提示准确率的可信度。

    标签平滑统一由传入的 criterion 负责，混合增强分支也走 criterion，保证两条路径口径一致。
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    augmented_batches = 0
    num_batches = len(loader)
    sample_counts = batch_sample_counts(loader)
    current_group_start = -1                       # 当前累积组起点（缓存组内样本总数用）
    current_group_samples = 1
    optimizer.zero_grad(set_to_none=True)          # 梯度累积的起点
    autocast_enabled = bool(use_bf16) and device.type == 'cuda'
    loop = tqdm(loader, desc=f'Epoch {epoch} Train')

    for batch_idx, (inputs, targets) in enumerate(loop):
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        # -------- 决定本 batch 使用哪种增强 --------
        # 开一种或两种，都按各自概率抽样（否则只开一种时概率参数会失效）。
        apply_mixup = False
        apply_cutmix = False
        r = np.random.rand()
        if mixup_enable and cutmix_enable:
            if r < mixup_prob:
                apply_mixup = True
            elif r < mixup_prob + cutmix_prob:
                apply_cutmix = True
        elif mixup_enable:
            apply_mixup = r < mixup_prob
        elif cutmix_enable:
            apply_cutmix = r < cutmix_prob

        if apply_mixup:
            inputs, targets_a, targets_b, lam = mixup_data(inputs, targets, mixup_alpha)
        elif apply_cutmix:
            inputs, targets_a, targets_b, lam = cutmix_data(inputs, targets, cutmix_alpha)

        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=autocast_enabled):
            outputs = model(inputs)
            if apply_mixup or apply_cutmix:
                # 直接用 criterion（它已经带 label_smoothing），与普通路径保持一致
                loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)
            else:
                loss = criterion(outputs, targets)
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()

        if apply_mixup or apply_cutmix:
            augmented_batches += 1

        # -------- 记录缩放前的真实平均损失 --------
        raw_loss = loss.item()
        running_loss += raw_loss * inputs.size(0)

        # -------- 梯度累积：组内按样本数加权 --------
        # 每个 batch 的权重 = 它贡献的样本数 / 组内总样本数，组内权重和恒为 1，
        # 最后那个不足整批的小 batch 会被正确降权（GRAD_ACCUM_STEPS=1 时权重就是 1）。
        # 注意：这只保证损失按样本数正确加权；模型里有 BatchNorm/Dropout，
        # 梯度累积并不等于真正的大 batch 前向，两者不会逐位一致。
        group_start = (batch_idx // grad_accum_steps) * grad_accum_steps
        if group_start != current_group_start:      # 同一个累积组只求一次总样本数
            current_group_start = group_start
            group_end = min(group_start + grad_accum_steps, num_batches)
            current_group_samples = sum(sample_counts[group_start:group_end])
        loss = loss * (sample_counts[batch_idx] / current_group_samples)
        loss.backward()

        # -------- 每个累积组（或最后一步）更新一次参数 --------
        if (batch_idx + 1) % grad_accum_steps == 0 or (batch_idx + 1) == num_batches:
            if grad_clip_max_norm and grad_clip_max_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_max_norm)
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            if ema is not None:
                ema.update_parameters(model)
            optimizer.zero_grad(set_to_none=True)

        if apply_mixup or apply_cutmix:
            loop.set_postfix(loss=raw_loss)
        else:
            loop.set_postfix(loss=raw_loss, acc=100. * correct / total if total > 0 else 0.)

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = 100.0 * correct / total if total > 0 else None
    return epoch_loss, epoch_acc, augmented_batches


def validate(model, loader, criterion, device, use_bf16=False):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    autocast_enabled = bool(use_bf16) and device.type == 'cuda'
    with torch.no_grad():
        loop = tqdm(loader, desc='Validation')
        for inputs, targets in loop:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=autocast_enabled):
                outputs = model(inputs)
                loss = criterion(outputs, targets)
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            loop.set_postfix(loss=loss.item(), acc=100. * correct / total)
    if total == 0:
        return float('nan'), 0.0
    return running_loss / total, 100.0 * correct / total


def check_image_sizes(dataset, max_check=200):
    """抽检图片尺寸：尺寸不一致且未设置 Resize 时，DataLoader 会在拼接 batch 时报错，提前给出提示。"""
    sizes = {}
    step = max(1, len(dataset) // max_check)
    for idx in range(0, len(dataset), step):
        img, _ = dataset[idx]
        if torch.is_tensor(img):
            size = tuple(img.shape[-2:])                 # (H, W)
        else:
            size = (img.size[1], img.size[0])            # PIL: (W, H) -> (H, W)
        sizes[size] = sizes.get(size, 0) + 1
        if len(sizes) > 3:
            break
    print(f"图片尺寸抽检（{sum(sizes.values())} 张, 记为 (H, W)）: {sizes}")
    return sizes


In [ ]:
# ==================== 超参数设置 ====================
DATA_DIR = r'C:\transformerFPGA\MobileViT\Dataset\Driver\train_p'
BATCH_SIZE = 16
EPOCHS = 30
LR = 0.0002
WEIGHT_DECAY = 0.05
PATIENCE = 10
NUM_WORKERS = 0
SAVE_PATH = 'mobilevitv2_0.75_10cls_relu.pth'   # 最佳模型（不要写成预训练权重文件名）
VAL_SPLIT = 0.2
SEED = 42

# ---- 模型结构：必须与预训练权重一致 ----
WIDTH_MULTIPLIER = 0.75
ACTIVATION = 'relu'                             # 原来是 swish(silu)
PRETRAINED_WEIGHTS = 'mobilevitv2-0.75.pt'      # ImageNet-1k 纯权重；1000 类分类头会被丢弃并重新初始化
RESIZE = None                                   # 数据本身是 256x256。注意：整数只固定"短边"，
                                                # 要真正固定尺寸请写 (256, 256)

# ---- 学习率调度：线性预热 + 余弦退火（下面两个是**绝对**学习率）----
WARMUP_EPOCHS = 3.0                             # 支持小数；设为 0 表示不预热
WARMUP_START_LR = 1e-6                          # 预热起点的绝对学习率
MAX_LR = LR                                     # 唯一配置入口是 LR；MAX_LR 只是"曲线峰值"的别名
T_MAX_ITERS = None                              # None = EPOCHS × 每轮优化步数；填数字时必须大于预热步数
ETA_MIN = 1e-10                                 # 退火终点的绝对学习率；超过周期后停在这里，不会回升

GRAD_CLIP_MAX_NORM = 5.0                        # 0 或 None 表示关闭
GRAD_ACCUM_STEPS = 1                            # 1 表示不启用梯度累积（必须 >= 1 的整数）
LABEL_SMOOTHING = 0.12                          # 只作用在 criterion 上，普通/混合增强两条路径共用

MIXUP_ENABLE = True
MIXUP_ALPHA = 0.2
CUTMIX_ENABLE = True
CUTMIX_ALPHA = 1.0
MIXUP_PROB = 0.01                               # 每个 batch 使用 MixUp 的概率（[0,1]）
CUTMIX_PROB = 0.05                              # 每个 batch 使用 CutMix 的概率（[0,1]）

# ---- 断点续训（与"从预训练权重初始化"是两件独立的事）----
RESUME = False
RESUME_PATH = 'checkpoint_epoch_30.pth'
ADDITIONAL_EPOCHS = 30                          # 续训时"再练多少轮"
RESUME_FROM_EMA = False                         # True = 用 EMA 权重继续训练（优化器状态与参数不完全对应）
RESUME_LR = None                                # 例如 0.0001；None 表示沿用检查点里的基准学习率
RESUME_EARLY_STOP_COUNTER = None                # None 表示沿用检查点里的计数值
RESUME_RESET_SCHEDULER = False                  # True = 按 ADDITIONAL_EPOCHS 重建 warmup+cosine（追加训练建议开）
RESUME_RESET_BEST_LOSS = False                  # True = 不继承历史最佳损失，续训重新判定"最佳"，
                                                # 这样新的 SAVE_PATH 才会被写入（追加训练常用）

# ---- EMA ----
EMA_ENABLE = True
EMA_MOMENTUM = 0.003

# ---- 检查点 ----
CHECKPOINT_EVERY_EPOCH = True
CHECKPOINT_PREFIX = 'checkpoint_epoch_'
CHECKPOINT_INCLUDE_EMA = True                   # 只影响每轮检查点是否带 EMA 权重；
                                                # 最佳模型始终保存"验证时用的那份权重"


def make_lr_lambda(warmup_steps, horizon, base_lr, start_lr, min_lr):
    """线性预热 + 余弦退火，返回"相对 base_lr 的倍率"（LambdaLR 会再乘 base_lr）。

    语义是**绝对学习率**：预热从 start_lr 升到 base_lr，再余弦降到 min_lr。
    因此换成别的基准（例如 RESUME_LR=1e-4）时，start_lr / min_lr 不会被等比缩放，
    仍然是配置里写的 1e-6 / 1e-10。

    退火进度用 ``min(1.0, …)`` 夹住：步数超过 horizon 后停在 min_lr，
    不会像 CosineAnnealingLR 那样在 T_max 之后周期性回升。
    ``warmup_steps=0`` 表示不预热，第一步就是基准学习率。
    """
    if base_lr <= 0:
        raise ValueError(f'基准学习率必须为正，当前为 {base_lr}')
    if min_lr > base_lr:
        raise ValueError(f'退火终点({min_lr})不能大于基准学习率({base_lr})')
    if horizon <= 0:
        raise ValueError(f'退火周期必须为正，当前为 {horizon}')
    # 必须满足 0 <= 预热 < 周期：否则整段训练都停在峰值
    # （例如周期=1、预热=1 时，学习率走完 1 步才刚到峰值，永远退不到终点）
    if not 0 <= warmup_steps < horizon:
        raise ValueError(f'预热步数必须在 [0, 退火周期) 内，当前 预热={warmup_steps}, 周期={horizon}')

    def lr_lambda(step):
        if step < warmup_steps:
            lr = start_lr + (base_lr - start_lr) * step / warmup_steps
        else:
            progress = min(1.0, (step - warmup_steps) / max(1, horizon - warmup_steps))
            lr = min_lr + (base_lr - min_lr) * 0.5 * (1.0 + math.cos(math.pi * progress))
        return lr / base_lr
    return lr_lambda


# ==================== 主流程 ====================
def main():
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'使用设备: {device}')

    use_bf16 = False
    if device.type == 'cuda':
        if torch.cuda.is_bf16_supported():
            use_bf16 = True
            print('bfloat16 混合精度训练已启用 (torch.autocast)')
        else:
            print('警告: 当前 GPU 不支持 bfloat16，回退为 FP32。')
    else:
        print('CPU 模式，使用 FP32 训练。')

    # ---------- 配置合法性 ----------
    if float(GRAD_ACCUM_STEPS) != int(GRAD_ACCUM_STEPS) or int(GRAD_ACCUM_STEPS) < 1:
        raise ValueError(f'GRAD_ACCUM_STEPS 必须是 >=1 的整数，当前为 {GRAD_ACCUM_STEPS}')
    grad_accum_steps = int(GRAD_ACCUM_STEPS)
    for prob_name, prob_value in (('MIXUP_PROB', MIXUP_PROB), ('CUTMIX_PROB', CUTMIX_PROB)):
        if not 0.0 <= float(prob_value) <= 1.0:
            raise ValueError(f'{prob_name} 必须在 [0, 1] 区间内，当前为 {prob_value}')
    if MIXUP_ENABLE and CUTMIX_ENABLE and MIXUP_PROB + CUTMIX_PROB > 1.0:
        print('警告: MIXUP_PROB + CUTMIX_PROB > 1，超出的部分不会生效'
              '（CutMix 实际概率被压到 1 - MIXUP_PROB）')

    # ---------- 数据准备 ----------
    transform = build_transform(RESIZE)
    # 预处理交给 ImageFolder，再用原生 Subset 切分（不要给 Subset 写子类补 transform：
    # Subset 的 __getitems__ 会绕过子类的 __getitem__，batch 里会混进 PIL 图像）。
    full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)
    num_classes = len(full_dataset.classes)
    print(f'数据集: {DATA_DIR}')
    print(f'类别数: {num_classes}, 类别名: {full_dataset.classes}')
    if num_classes != 10:
        print(f'提示: 数据集里有 {num_classes} 个类别目录，分类头按该数量构建。')
    check_image_sizes(full_dataset)
    if RESIZE is None:
        print('提示: RESIZE=None，要求所有图片尺寸一致（否则 DataLoader 拼接 batch 会报错）；'
              '若数据尺寸不统一，请把 RESIZE 设为 (256, 256)。')

    indices = list(range(len(full_dataset)))
    labels = [label for _, label in full_dataset.samples]

    train_indices, val_indices = train_test_split(
        indices, test_size=VAL_SPLIT, random_state=SEED, stratify=labels, shuffle=True
    )

    print(f"总样本数: {len(full_dataset)}")
    print(f"训练集样本数: {len(train_indices)}")
    print(f"验证集样本数: {len(val_indices)}")

    train_counts = Counter(labels[i] for i in train_indices)
    val_counts = Counter(labels[i] for i in val_indices)
    for class_idx, class_name in enumerate(full_dataset.classes):
        train_count = train_counts.get(class_idx, 0)
        val_count = val_counts.get(class_idx, 0)
        print(f"类别 '{class_name}': 训练集 {train_count} 张 "
              f"({train_count / len(train_indices) * 100:.2f}%), "
              f"验证集 {val_count} 张 ({val_count / len(val_indices) * 100:.2f}%)")

    train_dataset = Subset(full_dataset, train_indices)
    val_dataset = Subset(full_dataset, val_indices)

    pin = device.type == 'cuda'
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=pin)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=pin)

    # ---------- 步数与退火周期 ----------
    steps_per_epoch = math.ceil(len(train_loader) / grad_accum_steps)
    warmup_steps = int(round(WARMUP_EPOCHS * steps_per_epoch))
    if warmup_steps < 0:
        raise ValueError(f'WARMUP_EPOCHS 不能为负，当前为 {WARMUP_EPOCHS}')

    if T_MAX_ITERS is None:
        t_max_value = EPOCHS * steps_per_epoch
    else:
        t_max_value = int(T_MAX_ITERS)
    if t_max_value <= 0:
        raise ValueError(f'T_MAX_ITERS 必须为正整数，当前为 {T_MAX_ITERS}')
    print(f'每轮优化步数: {steps_per_epoch}, 预热步数: {warmup_steps}'
          + ('（WARMUP_EPOCHS=0，不预热）' if warmup_steps == 0 else '')
          + f', 按当前配置推算的退火周期: {t_max_value}')
    # 注意: "周期必须大于预热步数"的校验放到最终调度配置定稿之后再做 ——
    # 续训时周期可能来自检查点保存的配置或 ADDITIONAL_EPOCHS，
    # 用当前的 EPOCHS/WARMUP_EPOCHS 提前判断会误伤。

    # ---------- 模型 ----------
    model = MobileViTv2(build_opts(num_classes, WIDTH_MULTIPLIER, ACTIVATION)).to(device)
    print(f'模型: MobileViTv2 (宽度 {WIDTH_MULTIPLIER}x, 激活 {ACTIVATION}, {num_classes} 类), '
          f'参数量 {sum(p.numel() for p in model.parameters()):,}')

    if RESUME:
        # 续训时检查点里的权重才是最终结果：先加载预训练权重既浪费一次全量加载，
        # 又会让"预训练文件缺失"这种无关问题把续训卡住。
        print(f'RESUME=True：跳过预训练权重初始化（{PRETRAINED_WEIGHTS}），直接从检查点恢复')
    elif PRETRAINED_WEIGHTS:
        load_weights(model, PRETRAINED_WEIGHTS, expect_only_head_to_be_skipped=True)

    # ---------- 损失与优化器 ----------
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    no_decay_params = []
    decay_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.endswith('.bias') or len(param.shape) == 1:
            # 一维参数：所有 bias 以及 BN/LN 的 weight，不做权重衰减
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    print(f"应用权重衰减的参数数量: {len(decay_params)}")
    print(f"不应用权重衰减的参数数量: {len(no_decay_params)}")

    optimizer = optim.AdamW(
        [
            {'params': decay_params, 'weight_decay': WEIGHT_DECAY},
            {'params': no_decay_params, 'weight_decay': 0.0},
        ],
        lr=LR,
        betas=(0.9, 0.999),
    )

    # ---------- 本次实际使用的学习率调度配置 ----------
    # 续训时这些会被"检查点里保存的配置"覆盖，全部定稿后只创建一次调度器。
    sched_horizon = t_max_value
    sched_warmup = warmup_steps
    sched_base_lr = MAX_LR
    sched_start_lr = WARMUP_START_LR
    sched_min_lr = ETA_MIN
    restore_scheduler_state = None      # 非 None 表示"从检查点恢复调度器状态"

    def build_scheduler(horizon, warmup, base_lr, start_lr, min_lr):
        return torch.optim.lr_scheduler.LambdaLR(
            optimizer, lr_lambda=make_lr_lambda(warmup, horizon, base_lr, start_lr, min_lr)
        )

    def reset_scheduler():
        """按当前 sched_* 配置重建调度器，并把学习率放到曲线起点。

        创建调度器前要先写好 initial_lr/lr：调度器初始化时会自己算出当前步的学习率，
        不要在创建之后再改 lr（否则第一个优化步会直接用满学习率）。
        """
        for param_group in optimizer.param_groups:
            param_group['initial_lr'] = sched_base_lr
            param_group['lr'] = sched_base_lr
        return build_scheduler(sched_horizon, sched_warmup, sched_base_lr, sched_start_lr, sched_min_lr)

    # ---------- 保存路径与早停对象（先建好，续训时才能直接 load_state）----------
    save_path = SAVE_PATH
    if PRETRAINED_WEIGHTS and os.path.abspath(save_path) == os.path.abspath(PRETRAINED_WEIGHTS):
        raise ValueError(f'SAVE_PATH 不能与 PRETRAINED_WEIGHTS 相同（{save_path}），否则会覆盖 ImageNet 预训练权重')
    if RESUME and os.path.abspath(save_path) == os.path.abspath(RESUME_PATH):
        save_path = os.path.splitext(SAVE_PATH)[0] + '_best.pth'
        print(f'警告: SAVE_PATH 与 RESUME_PATH 相同，已改用 {save_path} 以免覆盖原检查点')
    if os.path.isfile(save_path):
        if RESUME and not RESUME_RESET_BEST_LOSS:
            print(f'注意: {save_path} 已存在。续训会继承历史最佳验证损失，只有比它更好时才会覆盖该文件。')
        else:
            print(f'注意: {save_path} 已存在。本次从零开始判定最佳（基准 val_loss=inf），'
                  f'第一轮只要拿到有限损失就会覆盖它。')

    early_stopping = EarlyStopping(patience=PATIENCE, verbose=True, path=save_path)

    # ---------- 断点续训 ----------
    start_epoch = 1
    best_val_acc = 0.0
    resume_ema_state = None
    pure_weight_resume = False

    if RESUME:
        if not os.path.isfile(RESUME_PATH):
            raise FileNotFoundError(f"RESUME=True 但找不到检查点: {RESUME_PATH}")
        print(f'从检查点恢复训练: {RESUME_PATH}')
        ckpt = torch.load(RESUME_PATH, map_location='cpu', weights_only=False)

        if isinstance(ckpt, dict) and ('model_state_dict' in ckpt or 'ema_model_state_dict' in ckpt):
            # ---- 1) 权重：续训要用"原始训练权重"（与 optimizer_state_dict 配套）----
            if RESUME_FROM_EMA and 'ema_model_state_dict' in ckpt:
                weight_key = 'ema_model_state_dict'
                print("  注意: RESUME_FROM_EMA=True，用 EMA 权重继续训练；优化器状态与参数不完全对应")
            elif 'model_state_dict' in ckpt:
                weight_key = 'model_state_dict'
            else:
                weight_key = 'ema_model_state_dict'
                print("  提示: 该检查点没有 model_state_dict，只能用 EMA 权重续训")
            model.load_state_dict(ckpt[weight_key], strict=True)
            print(f"  已恢复模型权重（来源: {weight_key}）")

            # ---- 2) 优化器状态 ----
            if ckpt.get('optimizer_state_dict'):
                try:
                    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
                except Exception as exc:
                    print(f"  警告: 优化器状态恢复失败（{type(exc).__name__}: {exc}），改为重新初始化优化器")

            # ---- 3) 学习率曲线：普通函数的闭包不会进 state_dict，所以要读回保存的整套配置 ----
            saved_sched = ckpt.get('scheduler_config') or {}
            if saved_sched:
                sched_horizon = int(saved_sched.get('horizon', sched_horizon))
                sched_warmup = int(saved_sched.get('warmup_steps', sched_warmup))
                sched_base_lr = float(saved_sched.get('base_lr', sched_base_lr))
                sched_start_lr = float(saved_sched.get('start_lr', sched_start_lr))
                sched_min_lr = float(saved_sched.get('min_lr', sched_min_lr))
                print(f"  采用检查点保存的调度配置: 周期 {sched_horizon} 步, 预热 {sched_warmup} 步, "
                      f"基准 {sched_base_lr}, 起点 {sched_start_lr}, 终点 {sched_min_lr}")
            else:
                print("  提示: 该检查点没有 scheduler_config（旧格式），曲线只能按当前配置恢复")
            if not RESUME_RESET_SCHEDULER:
                restore_scheduler_state = ckpt.get('scheduler_state_dict')

            # ---- 4) 训练进度、早停状态、EMA 权重、历史最佳的保存位置 ----
            start_epoch = int(ckpt.get('epoch', 0)) + 1
            best_val_acc = float(ckpt.get('best_val_acc', 0.0))
            early_stopping.load_state(ckpt)
            best_acc_txt = ('未知' if early_stopping.best_loss_val_acc is None
                            else f'{early_stopping.best_loss_val_acc:.2f}%')
            print(f"  将从 Epoch {start_epoch} 继续；历史最高验证准确率 {best_val_acc:.2f}%，"
                  f"早停基准 val_loss={early_stopping.best_loss:.4f}（第 {early_stopping.best_epoch} 轮，"
                  f"该轮 val_acc={best_acc_txt}），早停计数 {early_stopping.counter}/{PATIENCE}")
            resume_ema_state = ckpt.get('ema_model_state_dict')
            if early_stopping.best_checkpoint_path:
                print(f"  历史最佳权重实际保存在: {early_stopping.best_checkpoint_path}"
                      + ("（文件仍在）" if os.path.isfile(early_stopping.best_checkpoint_path)
                         else "（该文件目前不存在）"))
            else:
                print("  该检查点没有记录历史最佳权重的实际路径（旧格式）")

            if RESUME_RESET_BEST_LOSS:
                early_stopping.best_loss = np.inf
                early_stopping.best_epoch = None
                early_stopping.best_loss_val_acc = None
                print("  RESUME_RESET_BEST_LOSS=True：历史最佳损失已重置，续训会重新判定最佳并写入新的保存路径")
        else:
            # 纯权重文件：只加载参数（按形状过滤），优化器与调度器重新开始
            load_weights(model, RESUME_PATH, expect_only_head_to_be_skipped=False)
            pure_weight_resume = True
            print("  检测到纯权重文件：只加载参数；优化器重新初始化，学习率曲线按下面的设置重建。")

        # ---- 两种续训模式统一在这里定稿"本次的学习率与退火周期" ----
        if RESUME_RESET_SCHEDULER or pure_weight_resume:
            if T_MAX_ITERS is not None and not RESUME_RESET_SCHEDULER:
                sched_horizon = t_max_value          # 用户显式指定的周期优先
            else:
                # 追加训练的周期按"还要练多少轮"算（EPOCHS 是给全新训练用的）
                sched_horizon = max(1, int(ADDITIONAL_EPOCHS)) * steps_per_epoch
            sched_warmup = min(sched_warmup, max(1, int(round(0.1 * sched_horizon))))
            restore_scheduler_state = None
            print(f"  按本次训练重建曲线: 周期 {sched_horizon} 步（{ADDITIONAL_EPOCHS} 轮）, "
                  f"预热 {sched_warmup} 步")
        if RESUME_LR is not None:
            sched_base_lr = float(RESUME_LR)
            print(f"  基准学习率覆盖为 {sched_base_lr}")
        if RESUME_EARLY_STOP_COUNTER is not None:
            early_stopping.counter = int(RESUME_EARLY_STOP_COUNTER)
            print(f"  早停计数器已覆盖为: {early_stopping.counter}")

    # ---------- 最终调度配置校验（此时周期与预热都已定稿）----------
    if sched_warmup < 0:
        raise ValueError(f'预热步数不能为负，当前为 {sched_warmup}')
    if sched_warmup >= sched_horizon:
        if RESUME:
            clamped = max(0, sched_horizon - 1)
            print(f"  提示: 预热步数 {sched_warmup} 不小于退火周期 {sched_horizon}，已收缩为 {clamped}"
                  f"（必须满足 0 <= 预热 < 周期，否则整段训练都停在峰值）")
            sched_warmup = clamped
        else:
            raise ValueError(
                f"退火周期 ({sched_horizon} 步) 必须大于预热步数 ({sched_warmup} 步)："
                f"否则学习率整段都停在峰值、不会退火。可以把 WARMUP_EPOCHS 调小"
                f"（当前 {WARMUP_EPOCHS} 轮 ≈ {warmup_steps} 步），或把 EPOCHS / T_MAX_ITERS 调大。"
            )

    # ---------- 创建调度器（只创建一次），必要时恢复状态 ----------
    scheduler = reset_scheduler()

    if restore_scheduler_state:
        try:
            scheduler.load_state_dict(restore_scheduler_state)
            scheduler.base_lrs = [sched_base_lr for _ in scheduler.base_lrs]
            for param_group in optimizer.param_groups:
                param_group['lr'] = sched_base_lr * scheduler.lr_lambdas[0](scheduler.last_epoch)
            pos = int(getattr(scheduler, 'last_epoch', 0))
            print(f"  已恢复调度器状态: 第 {pos} 步, 当前学习率 {optimizer.param_groups[0]['lr']:.3e}")
            if pos >= sched_horizon:
                print(f"  注意: 已越过退火周期 {sched_horizon}，学习率会保持在终点 {sched_min_lr:g}（不会回升）；"
                      f"要重新退火请把 RESUME_RESET_SCHEDULER 设为 True")
        except Exception as exc:
            # 恢复失败时整体重建，避免对象处于"改了一半"的状态
            print(f"  警告: 调度器状态恢复失败（{type(exc).__name__}: {exc}），已重建调度器并从曲线起点开始")
            scheduler = reset_scheduler()
            print(f"  当前学习率 {optimizer.param_groups[0]['lr']:.3e}（曲线起点）")
    else:
        print(f"  学习率曲线: 周期 {sched_horizon} 步, 预热 {sched_warmup} 步, 基准 {sched_base_lr}, "
              f"起点 {sched_start_lr}, 终点 {sched_min_lr}；当前 {optimizer.param_groups[0]['lr']:.3e}")

    # ---------- EMA（必须在权重加载/恢复之后创建）----------
    ema = None
    if EMA_ENABLE:
        ema = EMA(model, ema_momentum=EMA_MOMENTUM, device=device)
        if resume_ema_state is not None:
            try:
                ema.ema_model.load_state_dict(resume_ema_state, strict=True)
                print("  EMA 已从检查点恢复（延续历史滑动平均，验证指标不会突变）")
            except Exception as exc:
                # 恢复失败时重建 EMA，确保它是"以当前权重为起点"的干净对象
                print(f"  警告: EMA 权重恢复失败（{type(exc).__name__}: {exc}），"
                      f"已重新以当前权重初始化 EMA")
                ema = EMA(model, ema_momentum=EMA_MOMENTUM, device=device)
        else:
            print(f"EMA 已启用，动量 = {EMA_MOMENTUM}（以当前权重为起点）")

    total_epochs = (start_epoch - 1 + ADDITIONAL_EPOCHS) if RESUME else EPOCHS

    # 记录**实际使用**的调度配置（而不是配置文件的原始值），否则续训记录会误导人
    scheduler_config = {
        'horizon': sched_horizon,
        'warmup_steps': sched_warmup,
        'base_lr': sched_base_lr,
        'start_lr': sched_start_lr,
        'min_lr': sched_min_lr,
        'steps_per_epoch': steps_per_epoch,
    }
    run_meta = {
        'num_classes': num_classes,
        'class_names': list(full_dataset.classes),
        'width_multiplier': WIDTH_MULTIPLIER,
        'activation': ACTIVATION,
        'resize': RESIZE,
        'data_dir': DATA_DIR,
        'val_split': VAL_SPLIT,
        'seed': SEED,
        'batch_size': BATCH_SIZE,
        'epochs_total': total_epochs,
        'ema_enabled': EMA_ENABLE,
        'ema_momentum': EMA_MOMENTUM if EMA_ENABLE else None,
        'label_smoothing': LABEL_SMOOTHING,
        'mixup': {'enable': MIXUP_ENABLE, 'alpha': MIXUP_ALPHA, 'prob': MIXUP_PROB},
        'cutmix': {'enable': CUTMIX_ENABLE, 'alpha': CUTMIX_ALPHA, 'prob': CUTMIX_PROB},
        # 实际使用的学习率曲线以顶层 scheduler_config 为准（唯一正式字段）；
        # 下面两项是按超参推算的原值，只用于对照。
        't_max_config': t_max_value,
        'warmup_steps_config': warmup_steps,
        'pretrained_weights': PRETRAINED_WEIGHTS if not RESUME else None,
        'torch_version': torch.__version__,
    }

    print(f'总训练轮次: {total_epochs}, 从 Epoch {start_epoch} 开始')
    print(f'MixUp 增强: {"开启" if MIXUP_ENABLE else "关闭"} (alpha={MIXUP_ALPHA}, prob={MIXUP_PROB})')
    print(f'CutMix 增强: {"开启" if CUTMIX_ENABLE else "关闭"} (alpha={CUTMIX_ALPHA}, prob={CUTMIX_PROB})')

    # ---------- 训练循环 ----------
    for epoch in range(start_epoch, total_epochs + 1):
        print(f"--- Epoch {epoch}/{total_epochs} (lr={optimizer.param_groups[0]['lr']:.3e}) ---")
        train_loss, train_acc, augmented = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, device, epoch, use_bf16,
            mixup_enable=MIXUP_ENABLE, mixup_alpha=MIXUP_ALPHA, mixup_prob=MIXUP_PROB,
            cutmix_enable=CUTMIX_ENABLE, cutmix_alpha=CUTMIX_ALPHA, cutmix_prob=CUTMIX_PROB,
            ema=ema, grad_accum_steps=grad_accum_steps,
            grad_clip_max_norm=GRAD_CLIP_MAX_NORM,
        )
        acc_note = f"{train_acc:.2f}%" if train_acc is not None else "n/a"
        if augmented:
            acc_note += f"（近似值：{augmented}/{len(train_loader)} 个 batch 做了混合增强）"
        print(f"Train Loss: {train_loss:.4f}, Train Acc(仅未增强样本): {acc_note}")

        eval_model = ema.ema_model if ema is not None else model
        val_loss, val_acc = validate(eval_model, val_loader, criterion, device, use_bf16)
        print(f"{'EMA ' if ema is not None else ''}Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        early_stopping(val_loss, val_acc, model, optimizer, scheduler, epoch, best_val_acc,
                       ema=ema, run_meta=run_meta, scheduler_config=scheduler_config)

        if CHECKPOINT_EVERY_EPOCH:
            checkpoint_state = build_checkpoint(
                model, ema, optimizer, scheduler, epoch, val_loss, val_acc, best_val_acc,
                early_stopping, run_meta, scheduler_config=scheduler_config,
                include_ema=CHECKPOINT_INCLUDE_EMA,
            )
            torch.save(checkpoint_state, f'{CHECKPOINT_PREFIX}{epoch}.pth')

        if early_stopping.early_stop:
            print("触发早停，结束训练。")
            break

    # ---------- 汇总（区分"本次是否写入"、"历史最佳"和"该检查点自己的指标"）----------
    print(f"\n训练完成: 历史最高验证准确率 {best_val_acc:.2f}%")
    if not early_stopping.saved_this_run:
        print(f"本次训练没有写入新的最佳检查点：验证损失从未低于继承的历史最佳 "
              f"{early_stopping.best_loss:.4f}（第 {early_stopping.best_epoch} 轮）")
        if RESUME:
            print(f"  注意: 续训用的 {RESUME_PATH} 里只有它自己那一轮的权重，"
                  f"并不包含第 {early_stopping.best_epoch} 轮的模型。")
            if early_stopping.best_checkpoint_path:
                print(f"  历史上那次最佳权重实际保存在: {early_stopping.best_checkpoint_path}"
                      + ("（文件仍在）" if os.path.isfile(early_stopping.best_checkpoint_path)
                         else "（该文件目前不存在）"))
            else:
                print("  没有记录历史最佳权重的实际路径（旧格式检查点），无法确认它现在还在不在。")
            print("  想让追加训练重新判定最佳并写入新文件，请设 RESUME_RESET_BEST_LOSS=True，"
                  "并把 SAVE_PATH 指到一个新文件名。")
        else:
            print("  这一轮训练里验证损失一次都没下降，请检查学习率、数据与模型配置。")
    else:
        own_acc = early_stopping.best_loss_val_acc
        own_acc_str = f", 该轮 val_acc={own_acc:.2f}%" if own_acc is not None else ", 该轮 val_acc=未知"
        print(f"本次最佳检查点: {save_path}（第 {early_stopping.best_epoch} 轮, "
              f"val_loss={early_stopping.best_loss:.4f}{own_acc_str}）")
        print("提示: 检查点按【最低验证损失】保存，它的准确率不一定等于历史最高准确率。")
        if ema is not None:
            print("推理/部署用检查点里的 ema_model_state_dict；续训用 model_state_dict + optimizer_state_dict。")

    # ---------- 收尾自检：把刚保存的权重读回来严格加载一遍 ----------
    if not early_stopping.saved_this_run:
        print(f"跳过收尾自检：本次没有写入 {save_path}，避免把上一次留下的旧文件当成新结果来检查。")
    else:
        try:
            ckpt = torch.load(save_path, map_location='cpu', weights_only=False)
            fresh = MobileViTv2(build_opts(num_classes, WIDTH_MULTIPLIER, ACTIVATION))
            fresh.load_state_dict(ckpt['model_state_dict'], strict=True)
            checked = 'model_state_dict'
            if 'ema_model_state_dict' in ckpt:
                fresh.load_state_dict(ckpt['ema_model_state_dict'], strict=True)
                checked += ' 与 ema_model_state_dict'
            cfg = ckpt.get('config', {})
            own_acc = ckpt.get('best_loss_val_acc')
            own_acc_str = f"{own_acc:.2f}%" if isinstance(own_acc, (int, float)) else "未知"
            print(f"自检通过: {save_path} 可被 {cfg.get('num_classes')} 类 / 宽度 {cfg.get('width_multiplier')} / "
                  f"激活 {cfg.get('activation')} 的模型严格加载（{checked}）；"
                  f"该检查点第 {ckpt.get('epoch')} 轮 val_loss={ckpt.get('val_loss')}, "
                  f"val_acc={ckpt.get('val_acc')}，早停最佳轮 val_acc={own_acc_str}")
            del fresh
        except Exception as exc:
            print(f"自检失败: {save_path} 重新加载出错 -> {type(exc).__name__}: {exc}")


if __name__ == '__main__':
    main()


使用设备: cuda
bfloat16 混合精度训练已启用 (torch.autocast)
数据集: C:\transformerFPGA\MobileViT\Dataset\Driver\train_p
类别数: 10, 类别名: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
图片尺寸抽检（201 张, 记为 (H, W)）: {(256, 256): 201}
提示: RESIZE=None，要求所有图片尺寸一致（否则 DataLoader 拼接 batch 会报错）；若数据尺寸不统一，请把 RESIZE 设为 (256, 256)。
总样本数: 22424
训练集样本数: 17939
验证集样本数: 4485
类别 'c0': 训练集 1991 张 (11.10%), 验证集 498 张 (11.10%)
类别 'c1': 训练集 1814 张 (10.11%), 验证集 453 张 (10.10%)
类别 'c2': 训练集 1853 张 (10.33%), 验证集 464 张 (10.35%)
类别 'c3': 训练集 1877 张 (10.46%), 验证集 469 张 (10.46%)
类别 'c4': 训练集 1861 张 (10.37%), 验证集 465 张 (10.37%)
类别 'c5': 训练集 1849 张 (10.31%), 验证集 463 张 (10.32%)
类别 'c6': 训练集 1860 张 (10.37%), 验证集 465 张 (10.37%)
类别 'c7': 训练集 1602 张 (8.93%), 验证集 400 张 (8.92%)
类别 'c8': 训练集 1529 张 (8.52%), 验证集 382 张 (8.52%)
类别 'c9': 训练集 1703 张 (9.49%), 验证集 426 张 (9.50%)
每轮优化步数: 1122, 预热步数: 3366, 按当前配置推算的退火周期: 33660
模型: MobileViTv2 (宽度 0.75x, 激活 relu, 10 类), 参数量 2,484,859
权重加载: mobilevitv2-0.75.pt
  已加载 267 个张量（模型共 269 个参数），跳过 2 个文件里

Epoch 1 Train: 100%|██████████| 1122/1122 [03:55<00:00,  4.76it/s, acc=16.6, loss=2.21]


Train Loss: 2.2658, Train Acc(仅未增强样本): 16.64%（近似值：73/1122 个 batch 做了混合增强）


Validation: 100%|██████████| 281/281 [00:18<00:00, 14.97it/s, acc=19.2, loss=2.25]


EMA Val Loss: 2.2356, Val Acc: 19.18%
验证损失下降，保存检查点至 mobilevitv2_0.75_10cls_relu.pth（第 1 轮: val_loss=2.2356, val_acc=19.18%；历史最高 val_acc=19.18%；推理权重=ema）
--- Epoch 2/30 (lr=6.733e-05) ---


...

Epoch 7 Train: 100%|██████████| 1122/1122 [03:23<00:00,  5.51it/s, acc=99.2, loss=0.735]


Train Loss: 0.6797, Train Acc(仅未增强样本): 99.15%（近似值：73/1122 个 batch 做了混合增强）


Validation: 100%|██████████| 281/281 [00:18<00:00, 15.21it/s, acc=99.4, loss=0.597]


EMA Val Loss: 0.6274, Val Acc: 99.42%
验证损失下降，保存检查点至 mobilevitv2_0.75_10cls_relu.pth（第 7 轮: val_loss=0.6274, val_acc=99.42%；历史最高 val_acc=99.42%；推理权重=ema）
--- Epoch 8/30 (lr=1.894e-04) ---


Epoch 8 Train:  37%|███▋      | 413/1122 [01:20<02:15,  5.23it/s, acc=99.5, loss=0.642]